# I tried coding the PIF myself.
### but its wrong
(I mixed up two equations there, either would have been):

# (a) analytic: one draw, no pre-sampled external input
p = lmbda + (1 - lmbda) * sigma * A / N
# (b) sampled OR, literally "external OR recurrent"
s = ext_t | (rng.random(N) < sigma * A / N)

In [ ]:
def PIF_memoryless_AA(
    N: int,
    sigma: float | int,
    T: np.number,
    h: float | int,
    init_cond: ArrayLike | None = None,
    seed: int | None = None,
):
    """
    self-recurrency is allowed here


    h: spikes per unit time
    lambda: probability of one spike in dt = 1
    since k ~ Poisson(h Delta_t)
    lambda = P(k≥1) = 1-P(k=0) = 1-e^{-hΔt}

    update rule:
    P(fire) = 1 - (not firing due to external input) * (not firing due to recurrent input)
    P(fire) = 1 - (1-lmbda)(1-V) 

    time step delta t = 1, implicit in discreet iteration through time
    """

    lmbda = 1 - np.exp(-h)
    w = sigma / (N)  # average weight of each connection
    t_axis = list(range(0, T))

    rng = np.random.default_rng(seed)

    # background activity
    external_activation = rng.binomial(n=1, p=lmbda, size=(N, len(t_axis)))

    if init_cond is None:
        # shuffle = F cause we only want the subset, order irrelevant
        selected_spike_at_start = rng.choice(
            a=N, size=int(lmbda * N), replace=False, shuffle=False
        )
        spiking_neurons = np.zeros(N)
        spiking_neurons[selected_spike_at_start] = 1
    else:
        spiking_neurons = init_cond

    recording = np.empty([N, len(t_axis)])

    for i, t in enumerate(t_axis):
        recurrent_activation = np.sum(spiking_neurons) * w
        ext_activation_t = external_activation[:, i]
        activation = (1 - lmbda) * recurrent_activation + lmbda * ext_activation_t
        spiking_neurons = rng.binomial(n=1, p=activation, size=N)
        recording[:, i] = spiking_neurons

    return t_axis, recording


t, recording = PIF_memoryless_AA(N=100, sigma=1, T=200, h=1, seed=1)
recording